# Day 12 Project: Searchable Knowledge Base

## What You're Building

You run this notebook with a folder of `.txt` files (five sample files are
provided in `sample_docs/`). The notebook ingests each file into a **persistent
Chroma collection**, embedding each document via `ollama.embeddings` with
`nomic-embed-text`. You then enter a search query at the prompt; the app embeds
the query, queries Chroma with an optional metadata filter (e.g. only files
tagged with a given category), and prints the top-3 matching document excerpts
with their filenames and similarity distances.

**That's the deliverable:** a local knowledge base you can query in natural
language, persisted to disk so re-runs skip re-embedding already-indexed files.

### Concepts this project composes

| Lesson | Concept used |
|--------|--------------|
| 1 | Why brute-force O(n) search doesn't scale — motivation for Chroma |
| 2 | `PersistentClient`, `get_or_create_collection`, `collection.add`, `collection.query` |
| 3 | `where`-clause metadata filters (`$eq`, `$in`) |
| 4 | `OllamaEmbeddingFunction` — Chroma embeds text automatically |
| 5 | `collection.upsert` (idempotent re-ingestion) and `collection.get` (inspection) |

> **Before you start:** complete Exercises 1–5, then come back here.

## Step 1 — Imports and configuration

Import everything you need and set the constants the rest of the notebook
will use (model name, store path, collection name, docs folder, top-k).

In [ ]:
import os
import chromadb
import ollama

# ── Configuration ────────────────────────────────────────────────────────────
EMBED_MODEL  = "nomic-embed-text"   # Ollama embedding model
STORE_PATH   = "./kb_store"         # directory where Chroma persists data
COLLECTION   = "knowledge_base"     # collection name inside the store
DOCS_FOLDER  = "./sample_docs"      # folder containing .txt source files
TOP_K        = 3                    # number of results to return per query

## Step 2 — Define the OllamaEmbeddingFunction

Subclass `chromadb.EmbeddingFunction` so Chroma can call Ollama automatically
on every `add` and `query` call — no manual embedding in application code.

```python
class OllamaEmbeddingFunction(chromadb.EmbeddingFunction):
    def __init__(self, model: str) -> None: ...
    def __call__(self, input: list[str]) -> list[list[float]]: ...
```

In [ ]:
class OllamaEmbeddingFunction(chromadb.EmbeddingFunction):
    """
    Wraps Ollama's embeddings endpoint as a Chroma EmbeddingFunction.

    Args:
        model: Ollama embedding model name (e.g. "nomic-embed-text").
    """

    def __init__(self, model: str = "nomic-embed-text") -> None:
        # TODO: store model as an instance attribute
        pass

    def __call__(self, input: list[str]) -> list[list[float]]:
        """
        Embed a batch of strings, returning one vector per string.

        Args:
            input: List of text strings to embed.

        Returns:
            List of embedding vectors (list[list[float]]).
        """
        # TODO: loop over input, call ollama.embeddings for each string,
        #       collect the "embedding" values, return the list of vectors
        pass

## Step 3 — Open (or create) the persistent collection

Use `chromadb.PersistentClient` with `STORE_PATH` and call
`get_or_create_collection` with your `OllamaEmbeddingFunction` attached.
This is the factory pattern from Lesson 4 — safe to call on every run.

In [ ]:
def open_collection(
    store_path: str,
    collection_name: str,
    embed_model: str,
):
    """
    Open a persistent Chroma collection with an embedded Ollama function.

    Args:
        store_path:      Path to the Chroma storage directory.
        collection_name: Name of the collection to open or create.
        embed_model:     Ollama model name for embeddings.

    Returns:
        A chromadb Collection with an OllamaEmbeddingFunction attached.
    """
    # TODO: create PersistentClient, instantiate OllamaEmbeddingFunction,
    #       call get_or_create_collection, return the collection
    pass


collection = open_collection(STORE_PATH, COLLECTION, EMBED_MODEL)
print(f"Collection '{collection.name}' open. Documents indexed: {collection.count()}")

## Step 4 — Ingest documents from the sample_docs folder

Walk `DOCS_FOLDER`, read every `.txt` file, and **upsert** it into the
collection (so re-runs are safe — no duplicate-ID errors).

Each document should carry metadata:
- `filename`: the basename of the file (e.g. `"ml_basics.txt"`)
- `category`: derived from the filename prefix before the first `_`
  (e.g. `"ml"` from `"ml_basics.txt"`)

Use the filename (without `.txt`) as the document ID.

Print a line for each new file and the final document count.

In [ ]:
def ingest_folder(collection, folder: str) -> int:
    """
    Upsert every .txt file in folder into the collection.

    Args:
        collection: Open Chroma collection.
        folder:     Path to the directory containing .txt files.

    Returns:
        Number of files ingested.
    """
    # TODO:
    # 1. List all .txt files in folder
    # 2. For each file:
    #    a. Read the text content
    #    b. Build the doc ID (filename stem, no extension)
    #    c. Build metadata: {"filename": ..., "category": ...}
    # 3. Call collection.upsert(ids=[...], documents=[...], metadatas=[...])
    #    (no embeddings= — OllamaEmbeddingFunction handles that)
    # 4. Return the number of files processed
    pass


ingested = ingest_folder(collection, DOCS_FOLDER)
print(f"Ingested {ingested} files. Total in collection: {collection.count()}")

## Step 5 — Search the knowledge base

Implement `search_kb` — it takes a plain-text query and an optional
`where` filter dict, queries the collection with `query_texts`, and
returns a list of result dicts:

```python
{"id": str, "filename": str, "category": str,
 "distance": float, "excerpt": str}
```

Then implement `print_results` to display them nicely.

In [ ]:
def search_kb(
    collection,
    query: str,
    top_k: int = 3,
    where: dict | None = None,
) -> list[dict]:
    """
    Semantic search over the knowledge base with optional metadata filter.

    Args:
        collection: Open Chroma collection.
        query:      Natural-language search query.
        top_k:      Number of results to return.
        where:      Optional Chroma where-clause dict (e.g. {"category": "ml"}).

    Returns:
        List of result dicts with keys id, filename, category, distance, excerpt.
    """
    # TODO:
    # 1. Build kwargs for collection.query:
    #    - query_texts=[query]  (Chroma embeds it via OllamaEmbeddingFunction)
    #    - n_results=top_k
    #    - include where only if it is not None
    # 2. Unpack results["ids"][0], results["documents"][0],
    #    results["distances"][0], results["metadatas"][0]
    # 3. Build and return the list of result dicts
    pass


def print_results(results: list[dict]) -> None:
    """Print search results in a readable format."""
    # TODO: for each result, print rank, distance, filename, and a
    #       truncated excerpt (first 200 chars)
    pass

## Step 6 — Interactive search loop

Run the interactive prompt. On each iteration:
1. Ask for a search query (empty input exits).
2. Ask for an optional category filter (empty = no filter).
3. Call `search_kb` and `print_results`.

The loop should continue until the user presses Enter with no query.

In [ ]:
# TODO: interactive search loop
#
# while True:
#     query = input("Search query (Enter to quit): ").strip()
#     if not query:
#         break
#     category = input("Filter by category (Enter to skip): ").strip()
#     where = {"category": category} if category else None
#     results = search_kb(collection, query, top_k=TOP_K, where=where)
#     print_results(results)

## Step 7 — Verify persistence

After the loop ends, use `collection.get` to inspect one stored document by
its ID, confirming the metadata (filename and category) survived on disk.
This demonstrates the Lesson 5 CRUD inspection pattern.

In [ ]:
# TODO: pick the ID of the first .txt file you ingested and call
#       collection.get(ids=[that_id]) — print the stored text and metadata
pass